In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_data import read_series
from src.research_validation import aligned_returns


# 01 Risk Free Data

Run after 01_Stock_Data. This module freezes the rate and benchmark conventions used by all subsequent modules.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load frozen rates and benchmark

Rates are treated as annual-effective decimals. The default benchmark is the S&P 500 price index, not SPY or a total-return index.


In [ ]:
risk_free_rates = read_series("data/inputs/risk_free_rates.parquet")
benchmark = read_series("data/inputs/sp500_prices.parquet")
if (risk_free_rates <= -1).any():
    raise ValueError("Annual-effective decimal rates must be greater than -1.")
display(risk_free_rates.describe())
display(benchmark.head())


## 3. Verify coverage and save

Check coverage before expensive pair screening. The same prior-session daily rate conversion is used later in Modules 09 and 12.


In [ ]:
test_prices = pd.read_parquet("test_prices.parquet")
coverage = aligned_returns(
    pd.DataFrame({"equity": cfg.initial_capital}, index=test_prices.index),
    benchmark,
    risk_free_rates,
)
risk_free_rates.to_frame("risk_free_rate").to_parquet("risk_free_rates.parquet")
benchmark.to_frame("benchmark").to_parquet("benchmark_prices.parquet")
display(coverage[["market_return", "rf_daily"]].head())
risk_free_rates.plot(figsize=(10, 3), title="Frozen annual risk free rate")
plt.show()
